## 2. 数据集准备

评测 RAG 应用，数据集必须有：

- 运行输入：
    - question[str]：问题
- 运行输出
    - answer[str]：RAG 应用给出的回答
    - contexts[list[str]]: 检索到的上下文，顺序则代表相似度。
- 评估输入
    - reference_context[str]: 参考上下文，用于评估检索的正确性。
    - reference_answer[str]: 参考答案，用于评估回答的正确性。

数据集有三种方法准备：

1. 使用开源数据集，比如 ragas 引用的 explodinggradients/fiqa 数据集。但是中文的 RAG 数据集较少。而且开源数据集只能代表 RAG 的通用能力，不能代表 RAG 在特定领域的能力。
2. 人工标注，这种方法需要大量的人力成本，但是可以标注特定领域的数据集。
3. 使用 LLM 自动抽取 QA 对，从而形成数据集。这种方法的优点是成本低，缺点是数据集的质量可能不高。自动标注尽量使用能力较强的 LLM，比如 GPT-4 等。

### 2.1 使用开源数据集

#### 2.1.1 cmrc 数据集下载和转换

正式项目中，应该以全部的数据作为我们的向量索引的基础数据，单条数据格式如上。我们挑选其中的 context 嵌入到向量索引中。

本次我们就只以 test 的 1000 条数据建立索引。同时以 test 的随机 50 条查询作为验证数据集。

此时我们可以准备两种数据集

1. 用于运行的数据集，即只有 question / reference_answer / reference_contexts 的数据集。
2. 用于评估的数据集，即使用 chain 检索后的数据集，包含全部数据。

前者在启动评估任务的时候还要注册 Provider 用于 RAG 生成，为模拟真实的情况，我们选择 1。那么此处只需要生成用于运行的数据集即可。


In [1]:
import json
import os
import pandas as pd

data_dir = 'cmrcData/data'
cmrc = {}
cmrc['train'] = pd.read_json(os.path.join(data_dir, 'cmrc2018_train.json'))
cmrc['test'] = pd.read_json(os.path.join(data_dir, 'cmrc2018_trial.json'))
cmrc['dev'] = pd.read_json(os.path.join(data_dir, 'cmrc2018_dev.json'))

In [2]:
cmrc['train']

,context_id,context_text,qas,title
0,TRAIN_186,范廷颂枢机（，），圣名保禄·若瑟（），是越南罗马天主教枢机。1963年被任为主教；1990年...,"[{'query_id': 'TRAIN_186_QUERY_0', 'query_text...",范廷颂
1,TRAIN_54,安雅·罗素法（，），来自俄罗斯圣彼得堡的模特儿。她是《全美超级模特儿新秀大赛》第十季的亚军。...,"[{'query_id': 'TRAIN_54_QUERY_0', 'query_text'...",安雅·罗素法
2,TRAIN_756,为日本漫画足球小将翼的一个角色，自小父母离异，与父亲一起四处为家，每个地方也是待一会便离开，...,"[{'query_id': 'TRAIN_756_QUERY_0', 'query_text...",岬太郎
3,TRAIN_36,NGC 6231是一个位于天蝎座的疏散星团，天球座标为赤经16时54分，赤纬-41度48分，...,"[{'query_id': 'TRAIN_36_QUERY_0', 'query_text'...",NGC 6231
4,TRAIN_573,国际初中科学奥林匹克（International Junior Science Olympi...,"[{'query_id': 'TRAIN_573_QUERY_0', 'query_text...",国际初中科学奥林匹克
...,...,...,...,...
2398,TRAIN_3816,布莱恩·杰伊·辛格（，），是一位美国电影导演、电影监制、编剧及演员，为制片公司的创始人。以1...,"[{'query_text': '布莱恩·杰伊·辛格的身份是什么？', 'query_id'...",布莱恩·辛格
2399,TRAIN_3817,圣若望广场（）是佛罗伦萨的一个广场，得名于圣若望洗礼堂，实际上是主教座堂广场的向西延伸。目前...,"[{'query_text': '圣若望广场位于哪里？', 'query_id': 'TRA...",圣若望广场 (佛罗伦萨)
2400,TRAIN_3818,《雾之恋》是香港歌手谭咏麟发行第七张粤语专辑，籍著上一年的专辑《春... 迟来的春天》的成功...,"[{'query_text': '《雾之恋》是香港歌手谁发行的粤语专辑？', 'query_...",雾之恋
2401,TRAIN_3820,里卡多·阿尔贝托·马蒂内利·贝罗卡尔（，），巴拿马政治家和企业家，前巴拿马总统。1952年3...,"[{'query_text': '里卡多·阿尔贝托·马蒂内利·贝罗卡尔身份是什么？', 'q...",里卡多·马蒂内利


In [39]:
train_data = []
for _, row in cmrc['train'].iterrows():
    train_data.append({
        "text": row["context_text"],
        "title": row["title"],
    })

In [40]:
train_data

[{'text': '范廷颂枢机（，），圣名保禄·若瑟（），是越南罗马天主教枢机。1963年被任为主教；1990年被擢升为天主教河内总教区宗座署理；1994年被擢升为总主教，同年年底被擢升为枢机；2009年2月离世。范廷颂于1919年6月15日在越南宁平省天主教发艳教区出生；童年时接受良好教育后，被一位越南神父带到河内继续其学业。范廷颂于1940年在河内大修道院完成神学学业。范廷颂于1949年6月6日在河内的主教座堂晋铎；及后被派到圣女小德兰孤儿院服务。1950年代，范廷颂在河内堂区创建移民接待中心以收容到河内避战的难民。1954年，法越战争结束，越南民主共和国建都河内，当时很多天主教神职人员逃至越南的南方，但范廷颂仍然留在河内。翌年管理圣若望小修院；惟在1960年因捍卫修院的自由、自治及拒绝政府在修院设政治课的要求而被捕。1963年4月5日，教宗任命范廷颂为天主教北宁教区主教，同年8月15日就任；其牧铭为「我信天主的爱」。由于范廷颂被越南政府软禁差不多30年，因此他无法到所属堂区进行牧灵工作而专注研读等工作。范廷颂除了面对战争、贫困、被当局迫害天主教会等问题外，也秘密恢复修院、创建女修会团体等。1990年，教宗若望保禄二世在同年6月18日擢升范廷颂为天主教河内总教区宗座署理以填补该教区总主教的空缺。1994年3月23日，范廷颂被教宗若望保禄二世擢升为天主教河内总教区总主教并兼天主教谅山教区宗座署理；同年11月26日，若望保禄二世擢升范廷颂为枢机。范廷颂在1995年至2001年期间出任天主教越南主教团主席。2003年4月26日，教宗若望保禄二世任命天主教谅山教区兼天主教高平教区吴光杰主教为天主教河内总教区署理主教；及至2005年2月19日，范廷颂因获批辞去总主教职务而荣休；吴光杰同日真除天主教河内总教区总主教职务。范廷颂于2009年2月22日清晨在河内离世，享年89岁；其葬礼于同月26日上午在天主教河内总教区总主教座堂举行。',
  'title': '范廷颂'},
 {'text': '安雅·罗素法（，），来自俄罗斯圣彼得堡的模特儿。她是《全美超级模特儿新秀大赛》第十季的亚军。2008年，安雅宣布改回出生时的名字：安雅·罗素法（Anya Rozova），在此之前是使用安雅·冈（）。安雅于俄罗斯出生，后来被一个居住在美国夏威夷群岛欧胡岛檀香山的家庭领养。安雅十七岁时

In [46]:
from config import MilvusConfig
from src.services.storage.milvus_client import MilvusExecutor
from src.services.data_load.data_storage import DataDBStorage
from langchain_core.documents import Document

class DataStorage(DataDBStorage):
    def __init__(self,):
        super().__init__()
        # 向量数据库
        self.vector = MilvusExecutor(MilvusConfig(collection_name='cmrc_dataset'))

    async def save_to_vector(self):
        documents = []
        for item in train_data:
            documents.append(Document(page_content=item['text'], title=item['title']))
        await self.vector.client.aadd_documents(documents=documents)


if __name__ == '__main__':
    await DataStorage().save_to_vector()

In [8]:
test_data = []
for _,row in cmrc['train'].iterrows():
    qas = row['qas']
    test_data.append({
        "text": row['context_text'],
        "qas": [{'question':_['query_text'],'answer':"\n".join([str(s) for s in _['answers']])} for _ in qas]
    })

In [9]:
test_data

[{'text': '范廷颂枢机（，），圣名保禄·若瑟（），是越南罗马天主教枢机。1963年被任为主教；1990年被擢升为天主教河内总教区宗座署理；1994年被擢升为总主教，同年年底被擢升为枢机；2009年2月离世。范廷颂于1919年6月15日在越南宁平省天主教发艳教区出生；童年时接受良好教育后，被一位越南神父带到河内继续其学业。范廷颂于1940年在河内大修道院完成神学学业。范廷颂于1949年6月6日在河内的主教座堂晋铎；及后被派到圣女小德兰孤儿院服务。1950年代，范廷颂在河内堂区创建移民接待中心以收容到河内避战的难民。1954年，法越战争结束，越南民主共和国建都河内，当时很多天主教神职人员逃至越南的南方，但范廷颂仍然留在河内。翌年管理圣若望小修院；惟在1960年因捍卫修院的自由、自治及拒绝政府在修院设政治课的要求而被捕。1963年4月5日，教宗任命范廷颂为天主教北宁教区主教，同年8月15日就任；其牧铭为「我信天主的爱」。由于范廷颂被越南政府软禁差不多30年，因此他无法到所属堂区进行牧灵工作而专注研读等工作。范廷颂除了面对战争、贫困、被当局迫害天主教会等问题外，也秘密恢复修院、创建女修会团体等。1990年，教宗若望保禄二世在同年6月18日擢升范廷颂为天主教河内总教区宗座署理以填补该教区总主教的空缺。1994年3月23日，范廷颂被教宗若望保禄二世擢升为天主教河内总教区总主教并兼天主教谅山教区宗座署理；同年11月26日，若望保禄二世擢升范廷颂为枢机。范廷颂在1995年至2001年期间出任天主教越南主教团主席。2003年4月26日，教宗若望保禄二世任命天主教谅山教区兼天主教高平教区吴光杰主教为天主教河内总教区署理主教；及至2005年2月19日，范廷颂因获批辞去总主教职务而荣休；吴光杰同日真除天主教河内总教区总主教职务。范廷颂于2009年2月22日清晨在河内离世，享年89岁；其葬礼于同月26日上午在天主教河内总教区总主教座堂举行。',
  'qas': [{'question': '范廷颂是什么时候被任为主教的？', 'answer': '1963年'},
   {'question': '1990年，范廷颂担任什么职务？', 'answer': '1990年被擢升为天主教河内总教区宗座署理'},
   {'question': '范廷颂是于何时何地出生的？', 'ans

In [10]:
with open('test_row_data.json', 'w') as f:
    json.dump(test_data, f, ensure_ascii=False, indent=4)